'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


In [ ]:
import os
from datetime import datetime, date
import random, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from scipy import sparse
from collections import defaultdict
import pandas as pd
import networkx as nx
import copy
import gzip
import pickle
from scipy.stats import rankdata
import time

### single concept's citation features

In [ ]:
time_start = time.time()
data_folder="data_concept_graph"
concept_folder="concept_folder"
# # Read all concepts together with time, citation information
dynamic_concept_file=os.path.join(data_folder,"full_dynamic_concept.parquet")
full_concepts_dynamic_data = pd.read_parquet(dynamic_concept_file)

# Read all concepts from full_concepts_for_openalex.txt
concepts_files = os.path.join(concept_folder, 'final_concepts.txt')
with open(concepts_files, 'r') as file:
    full_concepts = [concept.strip() for concept in file.readlines()]

print(f"Done, elapsed_time: {time.time() - time_start}\n full_concepts_dynamic_data: {len(full_concepts_dynamic_data)};\n full_concept: {len(full_concepts)}")


In [ ]:
NUM_OF_VERTICES=len(full_concepts)
vertex_degree_cutoff=1
years_delta=3
min_edges=1

In [ ]:
  # Comment
print("... columns:", full_concepts_dynamic_data.columns.tolist())

  # Comment
year_cols = [col for col in full_concepts_dynamic_data.columns if col.startswith('c') and col[1:].isdigit()]
print("... columns:", year_cols)
print("...:", [int(col[1:]) for col in year_cols])

  # Comment
print("...:", full_concepts_dynamic_data.shape)

In [ ]:

years=[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

day_origin = date(1990,1,1)
all_concepts_df = pd.DataFrame({'v1': range(0, NUM_OF_VERTICES)})

store_folder="data_for_features"
if not os.path.exists(store_folder):
    os.makedirs(store_folder)

start_time=time.time()
for yy in years:  
    print(f'Year: {yy}')
    day_curr=(date(yy,12,31)- day_origin).days
    columns_to_subtract = [f'c{i}' for i in range(2023, yy, -1)]
    print(columns_to_subtract)
    cols_to_sum = [f'c{i}' for i in range(yy, yy-years_delta, -1)]
    print(cols_to_sum)
    
    dynamic_concepts=full_concepts_dynamic_data[full_concepts_dynamic_data['time']<=day_curr]
    dynamic_concepts_df = dynamic_concepts.copy()
    
    dynamic_concepts_df[f'ct_{yy}'] = dynamic_concepts_df['ct'] - dynamic_concepts_df[columns_to_subtract].sum(axis=1)
    
    dynamic_concepts_df['ct_delta'] = dynamic_concepts_df[cols_to_sum].sum(axis=1)
    
    dynamic_concepts_df=dynamic_concepts_df[['v1', f'c{yy}', f'ct_{yy}', 'ct_delta']]
    
    dynamic_concepts_grouped = dynamic_concepts_df.groupby('v1').agg({f'c{yy}':'sum', f'ct_{yy}':'sum', 'ct_delta':'sum', 'v1':'size'}).rename(columns={'v1':f'num'}).reset_index()
    
    dynamic_concepts_grouped[f'c{yy}_m'] = dynamic_concepts_grouped[f'c{yy}'] / dynamic_concepts_grouped[f'num']
    dynamic_concepts_grouped[f'ct_{yy}_m'] = dynamic_concepts_grouped[f'ct_{yy}'] / dynamic_concepts_grouped[f'num']
    dynamic_concepts_grouped[f'ct_delta_m'] = dynamic_concepts_grouped['ct_delta'] / dynamic_concepts_grouped[f'num']
     
    
    # Merge with all_concepts_df
    dynamic_concepts_data = pd.merge(all_concepts_df, dynamic_concepts_grouped, on='v1', how='left')
    dynamic_concepts_data.fillna(0, inplace=True) # Fill NaN values with 0
    dynamic_concepts_data.sort_values(by='v1')
    
    data_file = os.path.join(store_folder, f"concept_node_citation_data_{yy}.parquet")
    dynamic_concepts_data.to_parquet(data_file, compression='gzip')
    print(f"in {yy}; time: {time.time()-start_time}\n")
    start_time=time.time()


In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import date

# ConfigurationParameters
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
years_delta = 3  # Code comment
day_origin = date(1990, 1, 1)

# Code comment
  # Comment
  # Comment

  # Comment
store_folder = "data_for_features"
if not os.path.exists(store_folder):
    os.makedirs(store_folder)

# Code comment
all_concepts_df = pd.DataFrame({'v1': range(0, NUM_OF_VERTICES)})

  # Comment
print("="*60)
print("...")
print("="*60)
print(f"...: {full_concepts_dynamic_data.shape}")
print(f"... columns: {full_concepts_dynamic_data.columns.tolist()}")

# Code comment
year_cols = [col for col in full_concepts_dynamic_data.columns 
             if col.startswith('c') and col[1:].isdigit()]
available_years = sorted([int(col[1:]) for col in year_cols])
print(f"... columns: {available_years}")
print(f"...: {max(available_years) if available_years else '...'}")
print("="*60)

start_time = time.time()
processed_years = []

for yy in years:
    print(f'\nProcess...: {yy}')
    chunk_start = time.time()
    
      # Comment
    day_curr = (date(yy, 12, 31) - day_origin).days
    
    # Code comment
    dynamic_concepts = full_concepts_dynamic_data[full_concepts_dynamic_data['time'] <= day_curr].copy()
    print(f"  ...Data volume: {len(dynamic_concepts)}  rows")
    
    if len(dynamic_concepts) == 0:
        print(f"  Warning: {yy}...Skip")
        continue
    
      # Comment
    # Code comment
    max_year = max(available_years) if available_years else yy
    future_years = [y for y in range(max_year, yy, -1) if y in available_years]
    columns_to_subtract = [f'c{y}' for y in future_years]
    
    print(f"  ...: {future_years}")
    
      # Comment
    if columns_to_subtract:
          # Comment
        missing_cols = [col for col in columns_to_subtract if col not in dynamic_concepts.columns]
        if missing_cols:
            print(f"  Warning: ... columnsdoes not exist: {missing_cols}")
            columns_to_subtract = [col for col in columns_to_subtract if col in dynamic_concepts.columns]
        
        if columns_to_subtract:
            dynamic_concepts[f'ct_{yy}'] = dynamic_concepts['ct'] - dynamic_concepts[columns_to_subtract].sum(axis=1)
        else:
            dynamic_concepts[f'ct_{yy}'] = dynamic_concepts['ct']
    else:
        dynamic_concepts[f'ct_{yy}'] = dynamic_concepts['ct']
    
      # Comment
    # Code comment
    past_years = [y for y in range(yy, yy - years_delta, -1) if y in available_years]
    cols_to_sum = [f'c{y}' for y in past_years]
    
    print(f"  ...: {past_years}")
    
      # Comment
    if cols_to_sum:
          # Comment
        missing_cols = [col for col in cols_to_sum if col not in dynamic_concepts.columns]
        if missing_cols:
            print(f"  Warning: ... columnsdoes not exist: {missing_cols}")
            cols_to_sum = [col for col in cols_to_sum if col in dynamic_concepts.columns]
        
        if cols_to_sum:
            dynamic_concepts['ct_delta'] = dynamic_concepts[cols_to_sum].sum(axis=1)
        else:
            dynamic_concepts['ct_delta'] = 0
    else:
        dynamic_concepts['ct_delta'] = 0
    
      # Comment
    c_col = f'c{yy}'
    if c_col not in dynamic_concepts.columns:
        print(f"  Warning: {c_col}  columnsdoes not exist...Create... columns")
        dynamic_concepts[c_col] = 0
    
      # Comment
    required_cols = ['v1', c_col, f'ct_{yy}', 'ct_delta']
    existing_required = [col for col in required_cols if col in dynamic_concepts.columns]
    
    if len(existing_required) < len(required_cols):
        missing = set(required_cols) - set(existing_required)
        print(f"  Error: ... columns: {missing}")
        continue
    
    dynamic_concepts_subset = dynamic_concepts[required_cols].copy()
    
      # Comment
    print(f"  Start......")
    agg_dict = {
        c_col: 'sum',
        f'ct_{yy}': 'sum',
        'ct_delta': 'sum',
        'v1': 'size'
    }
    
    dynamic_concepts_grouped = dynamic_concepts_subset.groupby('v1').agg(agg_dict).rename(columns={'v1': 'num'}).reset_index()
    
      # Comment
    # Code comment
    dynamic_concepts_grouped['num'] = dynamic_concepts_grouped['num'].replace(0, 1)
    
    dynamic_concepts_grouped[f'{c_col}_m'] = dynamic_concepts_grouped[c_col] / dynamic_concepts_grouped['num']
    dynamic_concepts_grouped[f'ct_{yy}_m'] = dynamic_concepts_grouped[f'ct_{yy}'] / dynamic_concepts_grouped['num']
    dynamic_concepts_grouped['ct_delta_m'] = dynamic_concepts_grouped['ct_delta'] / dynamic_concepts_grouped['num']
    
      # Comment
    print(f"  Merge......")
    dynamic_concepts_data = pd.merge(all_concepts_df, dynamic_concepts_grouped, on='v1', how='left')
    dynamic_concepts_data.fillna(0, inplace=True)
    dynamic_concepts_data.sort_values(by='v1', inplace=True)
    
    # ========== 8. SaveResult ==========
    data_file = os.path.join(store_folder, f"concept_node_citation_data_{yy}.parquet")
    dynamic_concepts_data.to_parquet(data_file, compression='gzip')
    
    processed_years.append(yy)
    
      # Comment
    elapsed = time.time() - chunk_start
    total_elapsed = time.time() - start_time
    print(f"  Done {yy}")
    print(f"    ...: {len(dynamic_concepts_grouped)}")
    print(f"    ...: {dynamic_concepts_data.shape}")
    print(f"    ...: {elapsed:.2f}s")
    print(f"    Cumulative...: {total_elapsed:.2f}s")

  # Comment
print("\n" + "="*60)
print("Processing complete...")
print("="*60)
print(f"SuccessfullyProcess...: {processed_years}")
print(f"Total time: {time.time() - start_time:.2f}s")
print(f"Output...: {store_folder}")

  # Comment
print("\nGenerate...:")
for file in sorted(os.listdir(store_folder)):
    if file.endswith('.parquet'):
        file_path = os.path.join(store_folder, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        print(f"  {file}: {file_size:.2f} MB")

print("="*60)

### concept pair's citation features

In [ ]:
time_start = time.time()
data_folder="data_concept_graph"

# Read all concepts together with time, citation information
graph_file=os.path.join(data_folder,"full_dynamic_graph.parquet")
full_edge_dynamic_data = pd.read_parquet(graph_file)

print(f"Done, elapsed_time: {time.time() - time_start}\n full_edge_dynamic_data: {len(full_edge_dynamic_data)};\n")


In [ ]:

years=[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,2023,2024, 2025]

day_origin = date(1990,1,1)
 
store_folder="data_for_features"
start_time=time.time()
for yy in years:  
    print(f'Year: {yy}')
    day_curr=(date(yy,12,31)- day_origin).days
    columns_to_subtract = [f'c{i}' for i in range(2023, yy, -1)]
    print(columns_to_subtract)
    cols_to_sum = [f'c{i}' for i in range(yy, yy-years_delta, -1)]
    print(cols_to_sum)
    
    dynamic_pairs=full_edge_dynamic_data[full_edge_dynamic_data['time']<=day_curr]
    dynamic_pairs_df = dynamic_pairs.copy()
    
    dynamic_pairs_df[f'ct_{yy}'] = dynamic_pairs_df['ct'] - dynamic_pairs_df[columns_to_subtract].sum(axis=1)
    
    dynamic_pairs_df['ct_delta'] = dynamic_pairs_df[cols_to_sum].sum(axis=1)
    
    dynamic_pairs_df=dynamic_pairs_df[['v1', 'v2', f'c{yy}', f'ct_{yy}', 'ct_delta']]
    
    dynamic_pairs_grouped = dynamic_pairs_df.groupby(['v1','v2']).agg({f'c{yy}':'sum', f'ct_{yy}':'sum', 'ct_delta':'sum', 'v1':'size'}).rename(columns={'v1':f'num'}).reset_index()
    
    dynamic_pairs_grouped[f'c{yy}_m'] = dynamic_pairs_grouped[f'c{yy}'] / dynamic_pairs_grouped[f'num']
    dynamic_pairs_grouped[f'ct_{yy}_m'] = dynamic_pairs_grouped[f'ct_{yy}'] / dynamic_pairs_grouped[f'num']
    dynamic_pairs_grouped[f'ct_delta_m'] = dynamic_pairs_grouped['ct_delta'] / dynamic_pairs_grouped[f'num']
    
    data_file = os.path.join(store_folder, f"concept_pair_citation_data_{yy}.parquet")
    dynamic_pairs_grouped.to_parquet(data_file, compression='gzip')
    print(f"in {yy}; time: {time.time()-start_time}\n")
    start_time=time.time()
    